In [1]:
import os

# add current directory to path
import sys

sys.path.append(os.path.dirname(os.getcwd()))

import wandb
from evaluation.analysis_helper import get_dataframes_from_run

api = wandb.Api()

wandb_project = "learneddb/lcm"
if os.path.exists("wandb_df_cache.pkl"):
    import pickle

    with open("wandb_df_cache.pkl", "rb") as f:
        df_cache = pickle.load(f)
else:
    raise FileNotFoundError(
        "wandb_df_cache.pkl not found. Are you really sure that there is no cache?"
    )
    df_cache = dict()

In [ ]:
run_id = "5dztgzue"
run = get_dataframes_from_run(run_id, df_cache, api, skip_cache=False)

# print available keys
print(f"Available keys in run: {list(run.keys())} ")
available_keys = list(run.keys())

wandb: Currently logged in as: jwehrstein (learneddb) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


Available keys in run: ['accesspath_eval_pred_table', 'accesspath_eval_w_aggregate_pred_table', 'join_order_job_eval_pred_table', 'join_order_job_extended_eval_pred_table', 'join_order_job_light_eval_pred_table', 'physop_eval_pred_table', 'physop_eval_with_idxs_pred_table'] 


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [4]:
from evaluation.analysis_helper import search_plan
import json
from ipywidgets import interact


@interact(key=available_keys)
def print_worst(key: str):
    eval_pred_table = run[key]

    # sort by q-error
    eval_pred_table = eval_pred_table.sort_values(by="qerror", ascending=False)

    # show the first 10 rows
    cols_to_show = ["label", "prediction", "sql", "hint"]
    eval_pred_table[cols_to_show].head(10).style.set_properties(
        **{"text-align": "left"}
    ).set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])

    worst_instance = eval_pred_table.iloc[0]
    sql = worst_instance["sql"]
    hint = worst_instance["hint"]

    # get query plan
    if key == "accesspath_eval_pred_table":
        dataset = "ap"
    elif key == "accesspath_eval_w_aggregate_pred_table":
        dataset = "ap_aggr"
    elif key == "physop_eval_pred_table":
        dataset = "physop"
    elif key == "physop_eval_with_idxs_pred_table":
        dataset = "physop_idx"
    elif key == "join_order_job_eval_pred_table":
        dataset = "job"
    elif key == "join_order_job_light_eval_pred_table":
        dataset = "job_light"
    else:
        raise ValueError(f"Unknown key: {key}")

    _, parsed_plan = search_plan(sql, hint, dataset=dataset)
    print(
        f"Label: {worst_instance['label']}, Prediction: {worst_instance['prediction']}, Q-Error: {worst_instance['qerror']}"
    )
    print(json.dumps(parsed_plan[0], indent=2))

interactive(children=(Dropdown(description='key', options=('accesspath_eval_pred_table', 'accesspath_eval_w_ag…